# Exploratory Data Analysis of the Vitamin D Transcriptomic Subset

## Data Loading and Initial Setup

In [ ]:
# Import core libraries
import os
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Write Parquet using pyarrow directly (preserves the gene index)
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
from pathlib import Path

FIGURES_DIR = Path("..") / "results" / "figures" / "paper" / "figure_1"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("..") / "results" / "dfs"

print("Saving figures to:", FIGURES_DIR.resolve())

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    "font.size": mpl.rcParams["font.size"] + 4
})

In [ ]:
# Load subset metadata and expression files
cell_meta = pd.read_csv("../data/exports/subset_cell_lines_meta.csv")
cmp_meta = pd.read_csv("../data/exports/subset_compounds_meta.csv")
gene_meta = pd.read_csv("../data/exports/subset_genes_meta.csv")
sig_meta = pd.read_csv("../data/exports/subset_signatures_meta.csv")
exp_df = pd.read_csv("../data/exports/subset_expression_wide_gene_id.csv", index_col=0)

In [ ]:
# Quick checks on dimensions
print("Cell lines:", cell_meta.shape)
print("Compounds:", cmp_meta.shape)
print("Genes:", gene_meta.shape)
print("Signatures:", sig_meta.shape)
print("Expression matrix:", exp_df.shape)

# Preview first rows
display(cell_meta.head())
display(cmp_meta.head())
display(sig_meta.head())
display(exp_df.iloc[:5, :5])


# Utility function for quick dataframe check
def quick_check(df, name, n=3):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Null values:\n", df.isna().sum().sort_values(ascending=False).head())
    print("Duplicates:", df.duplicated().sum())
    print("First rows:")
    display(df.head(n))
    print("\n")

# Run checks on metadata
quick_check(cell_meta, "Cell metadata")
quick_check(cmp_meta, "Compound metadata")
quick_check(gene_meta, "Gene metadata")
quick_check(sig_meta, "Signature metadata")

# Special check for expression matrix
print("--- Expression matrix ---")
print("Shape:", exp_df.shape)
print("Nulls (per signature):", (exp_df.isna().sum(axis=0) > 0).sum(), "/", exp_df.shape[1])
print("Nulls (per gene):", (exp_df.isna().sum(axis=1) > 0).sum(), "/", exp_df.shape[0])
display(exp_df.iloc[:5, :5])

### Cleaning step: remove empty signatures and realign metadata

In [ ]:
# --- Diagnose missing values ---
exp_numeric = exp_df.select_dtypes(include=[np.number])  # genes × signatures

nan_by_sig = exp_numeric.isna().sum(axis=0).sort_values(ascending=False)  # per signature (column)
nan_by_gene = exp_numeric.isna().sum(axis=1).sort_values(ascending=False) # per gene (row)

# Identify signatures with no NaNs (valid signatures)
valid_sig_cols = nan_by_sig[nan_by_sig == 0].index.tolist()

# Filter expression matrix to keep only valid signatures
exp_clean = exp_numeric[valid_sig_cols]   # shape: genes × valid_signatures
print("Expression matrix shape after cleaning:", exp_clean.shape)

# Align signature metadata to the cleaned matrix
sig_meta_clean = (
    sig_meta.set_index('sig_id')
    .reindex(valid_sig_cols)
    .reset_index()               # bring sig_id back as a column
    .rename(columns={"index":"sig_id"})  # ensure column name is consistent
)

# Sanity check
assert list(valid_sig_cols) == list(sig_meta_clean['sig_id'])
print("Signature metadata shape after cleaning:", sig_meta_clean.shape)

## Signature Distribution by Compound and Cell Line — setup
Build tidy summary tables for signatures per compound and per cell line

In [ ]:
# Map pert_id -> compound name
pert_lookup = cmp_meta[['pert_id', 'cmap_name']].drop_duplicates()

# --- Counts by compound (pert_id) ---
sig_by_pert = (
    sig_meta
    .groupby('pert_id', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .merge(pert_lookup, on='pert_id', how='left')
    .assign(cmap_name=lambda d: d['cmap_name'].fillna(d['pert_id']))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

# Count signatures grouped by compound name (cmap_name)
sig_by_cmap = (
    sig_meta
    .merge(cmp_meta[['pert_id', 'cmap_name']], on='pert_id', how='left')
    .groupby('cmap_name', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

display(sig_by_cmap)


# --- Counts by cell line ---
sig_by_cell = (
    sig_meta
    .groupby('cell_id', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

# Display summary tables
display(sig_by_pert)
display(sig_by_cell)


In [ ]:
# Create side-by-side barplots: compound vs. cell line
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios':[2,1]})

# --- Left: signatures per compound (cmap_name) ---
sns.barplot(
    data=sig_by_cmap,
    x='n_signatures',
    y='cmap_name',
    order=sig_by_cmap.sort_values('n_signatures', ascending=False)['cmap_name'],
    errorbar=None,
    ax=axes[0],
    color="steelblue"
)
axes[0].set_xlabel('Number of signatures')
axes[0].set_ylabel('Compound (cmap_name)')
axes[0].set_title('Signatures per compound')

# --- Right: signatures per cell line ---
sns.barplot(
    data=sig_by_cell,
    x='cell_id',
    y='n_signatures',
    order=sig_by_cell.sort_values('n_signatures', ascending=False)['cell_id'],
    errorbar=None,
    ax=axes[1],
    color="steelblue"
)
axes[1].set_xlabel('Cell line')
axes[1].set_ylabel('Number of signatures')
axes[1].set_title('Signatures per cell line')

plt.tight_layout()
plt.show()


---

## Global Distribution of Expression Values

In [ ]:
# Use only numeric expression columns (signatures)
exp_numeric = exp_df.select_dtypes(include=[np.number])

# Flatten to 1D numeric array
vals = exp_numeric.to_numpy().ravel()

# Create side-by-side panels: histogram and boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios':[3,1]})

# --- Left: histogram ---
sns.histplot(x=vals, bins=100, kde=True, ax=axes[0], color="green")
axes[0].set_xlabel("Z-score value")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of expression values")

# --- Right: boxplot ---
sns.boxplot(y=vals, ax=axes[1], color="yellow")
axes[1].set_ylabel("Z-score value")
axes[1].set_title("Global distribution (boxplot)")

plt.tight_layout()
plt.show()

---

## Validation of Perturbation Time and Dose

In [ ]:
# Check unique perturbation times
print("Unique perturbation times:", sig_meta['pert_time'].unique())
print("Unique time units:", sig_meta['pert_time_unit'].unique())

# Summarize doses by compound
dose_summary = (
    sig_meta
    .merge(cmp_meta[['pert_id','cmap_name']], on='pert_id', how='left')
    .groupby(['cmap_name','pert_dose','pert_dose_unit'], as_index=False)
    .agg(n_signatures=('sig_id','count'))
    .sort_values(['cmap_name','pert_dose'])
)

display(dose_summary)

plt.figure(figsize=(10,5))
sns.boxplot(
    data=sig_meta.merge(cmp_meta[['pert_id','cmap_name']], on='pert_id', how='left'),
    x='cmap_name',
    y='pert_dose',
    hue='cmap_name',         # use cmap_name as hue
    dodge=False,             # avoid side-by-side duplication
    legend=False,            # hide redundant legend
    palette="Set3"           # color palette
)
plt.yscale('log')
plt.xlabel("Compound (cmap_name)")
plt.ylabel("Dose (µM, log scale)")
plt.title("Dose distribution across compounds")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---

## Principal Component Analysis (PCA)

In [ ]:
# Transpose: signatures × genes
X = exp_clean.T  

# Scale data
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)

# Fit PCA (2 components for visualization)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Build results dataframe
pca_df = pd.DataFrame(X_pca, columns=['PC1','PC2'])
pca_df = pd.concat(
    [pca_df, sig_meta_clean[['sig_id','cell_id','pert_id']].reset_index(drop=True)],
    axis=1
).merge(cmp_meta[['pert_id','cmap_name']], on='pert_id', how='left')

# PCA plots: compounds vs cell lines, side by side
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# --- Left: PCA by compound ---
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2',
    hue='cmap_name', palette='Set2', s=60, alpha=0.85,
    ax=axes[0]
)
axes[0].set_title("PCA of Vitamin D signatures (by compound)")

# --- Right: PCA by cell line ---
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2',
    hue='cell_id', palette='tab10', s=60, alpha=0.85,
    ax=axes[1]
)
axes[1].set_title("PCA of Vitamin D signatures (by cell line)")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figure_1AB_pca.svg",
    bbox_inches="tight"
)
plt.show()

# Variance explained
print("Explained variance (PC1, PC2):", pca.explained_variance_ratio_)


In [ ]:
from sklearn.metrics import pairwise_distances

# Compute Euclidean distance matrix
dist_matrix = pairwise_distances(X_scaled, metric="euclidean")

dist_matrix.shape

In [ ]:
import numpy as np

print("NaNs en X_scaled:", np.isnan(X_scaled).sum())
print("NaNs en dist_matrix:", np.isnan(dist_matrix).sum())
print("Diagonal min/max:", dist_matrix.diagonal().min(), dist_matrix.diagonal().max())
print("Simetría (max abs diff):", np.max(np.abs(dist_matrix - dist_matrix.T)))

In [ ]:
import numpy as np
from skbio.stats.distance import DistanceMatrix

# Force exact symmetry and hollow diagonal
dist_matrix_sym = (dist_matrix + dist_matrix.T) / 2
np.fill_diagonal(dist_matrix_sym, 0.0)

ids = sig_meta_clean["sig_id"].astype(str).values
dm = DistanceMatrix(dist_matrix_sym, ids=ids)

dm

PERMANOVA for cell line

In [ ]:
from skbio.stats.distance import permanova

meta_perm = sig_meta_clean.copy()
meta_perm["sig_id"] = meta_perm["sig_id"].astype(str)
meta_perm = meta_perm.set_index("sig_id")

permanova_cell = permanova(dm, meta_perm["cell_id"], permutations=999)
permanova_cell

In [ ]:
F = permanova_cell["test statistic"]
g = permanova_cell["number of groups"]
N = permanova_cell["sample size"]

R2 = (F * (g - 1)) / (F * (g - 1) + (N - g))
R2

PERMANOVA for compound

In [ ]:
meta_perm_comp = sig_meta_clean.copy()
meta_perm_comp["sig_id"] = meta_perm_comp["sig_id"].astype(str)
meta_perm_comp = meta_perm_comp.set_index("sig_id")

permanova_comp = permanova(dm, meta_perm_comp["pert_id"], permutations=999)
permanova_comp

In [ ]:
F = permanova_comp["test statistic"]
g = permanova_comp["number of groups"]
N = permanova_comp["sample size"]

R2 = (F * (g - 1)) / (F * (g - 1) + (N - g))
R2

Additive PERMANOVA

In [ ]:
import pandas as pd
import numpy as np

# 1) Distance matrix as a labeled DataFrame
dist_df = pd.DataFrame(dist_matrix_sym, index=dm.ids, columns=dm.ids)
dist_path = RESULTS_DIR / "permanova_dist_matrix.csv"
dist_df.to_csv(dist_path)

# 2) Metadata aligned to the same ids
meta_df = sig_meta_clean.copy()
meta_df["sig_id"] = meta_df["sig_id"].astype(str)
meta_df = meta_df.set_index("sig_id").loc[dm.ids, ["cell_id", "pert_id", "pert_dose"]]

meta_path = RESULTS_DIR / "permanova_metadata.csv"
meta_df.to_csv(meta_path)

dist_path, meta_path

---

## Scree Plot: Explained Variance of Principal Components

In [ ]:
# Run PCA on the expression matrix (transpose so samples are rows)
pca = PCA()
pca.fit(exp_clean.T)

# Explained variance
explained_var = pca.explained_variance_ratio_ * 100  # in %
cumulative_var = np.cumsum(explained_var)

# Scree plot
plt.figure(figsize=(8,6))
plt.plot(range(1, len(explained_var)+1), explained_var, marker='o', label="Individual variance")
plt.plot(range(1, len(explained_var)+1), cumulative_var, marker='s', linestyle='--', label="Cumulative variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance (%)")
plt.title("Scree Plot of PCA")
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figure_1C_pca_scree_plot.svg",
    bbox_inches="tight"
)
plt.show()


---


In [ ]:
# Quantify how many PCs are needed to reach common variance thresholds
pca_full = PCA().fit(exp_clean.T)
explained = pca_full.explained_variance_ratio_
cum = np.cumsum(explained)

def pcs_for(threshold):
    return int(np.searchsorted(cum, threshold) + 1)

for thr in [0.50, 0.70, 0.80, 0.90, 0.95]:
    print(f"PCs to reach {int(thr*100)}% variance:", pcs_for(thr))


---


### Closing & Export of Cleaned Data

In [ ]:
# Create output folder if needed
out_dir = "../data/exports"
os.makedirs(out_dir, exist_ok=True)

# ---- Expression matrix (genes × signatures) ----
# Ensure column order matches metadata and set a clear index name
exp_clean = exp_clean.loc[:, sig_meta_clean['sig_id']]
exp_clean.index.name = 'gene_id'

table = pa.Table.from_pandas(exp_clean, preserve_index=True)
pq.write_table(table, f"{out_dir}/expression_matrix_clean.parquet", compression="snappy")

# ---- Signature metadata (aligned to exp_clean columns) ----
sig_meta_clean.to_csv(f"{out_dir}/signature_metadata_clean.csv", index=False)

print("✅ Cleaned data exported:",
      "\n- expression_matrix_clean.parquet",
      "\n- signature_metadata_clean.csv")